# Customer Segmentation with Actionable Business Insights

**Task 8 — Machine Learning Project**

---

## Objective
Build a customer segmentation solution that:
1. Groups customers with similar purchasing behaviour (K-Means Clustering)
2. Predicts total spending using Linear/Ridge Regression
3. Predicts purchase likelihood using Logistic Regression
4. Optimises models with hyperparameter tuning
5. Converts results into actionable business recommendations

---

## Step 1: Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

# Add project root to path
BASE_DIR = os.path.abspath('..')
sys.path.insert(0, BASE_DIR)

IMAGES_DIR  = os.path.join(BASE_DIR, 'images')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('All libraries imported successfully!')
print(f'pandas {pd.__version__} | numpy {np.__version__} | sklearn installed')

## Step 2: Load and Understand the Data

In [ ]:
# Load raw data
data_path = os.path.join(BASE_DIR, 'data', 'customer_data.csv')
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head(10)

In [ ]:
# Data types and basic info
df.info()

In [ ]:
# Statistical summary
df.describe().round(2)

In [ ]:
# Check for missing values and duplicates
print('Missing Values:')
print(df.isnull().sum())
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nGender Distribution:\n{df["Gender"].value_counts()}')
print(f'\nProduct Category:\n{df["ProductCategory"].value_counts()}')
print(f'\nPurchase Likelihood:\n{df["PurchaseLikelihood"].value_counts()}')

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Spending Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['TotalSpending'], bins=30, kde=True, color='#2196F3', ax=axes[0])
axes[0].set_title('Total Spending Distribution', fontweight='bold')
axes[0].set_xlabel('Total Spending ($)')

sns.histplot(df['PurchaseFrequency'], bins=25, kde=True, color='#4CAF50', ax=axes[1])
axes[1].set_title('Purchase Frequency Distribution', fontweight='bold')
axes[1].set_xlabel('Purchase Frequency')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'spending_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if c != 'CustomerID']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots for outlier detection
features = ['AnnualIncome', 'TotalSpending', 'PurchaseFrequency',
            'AverageOrderValue', 'DaysSinceLastPurchase', 'WebsiteVisits']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, (col, color) in enumerate(zip(features, colors)):
    sns.boxplot(y=df[col], ax=axes[i//3][i%3], color=color)
    axes[i//3][i%3].set_title(col, fontweight='bold')

fig.suptitle('Outlier Detection — Boxplots', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'boxplot_outliers.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Average Spending by Category
avg_spending = df.groupby('ProductCategory')['TotalSpending'].mean().sort_values(ascending=True)
palette = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(avg_spending.index, avg_spending.values, color=palette)
for bar, val in zip(bars, avg_spending.values):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontweight='bold')
ax.set_title('Average Spending by Product Category', fontweight='bold')
ax.set_xlabel('Average Total Spending ($)')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'average_spending.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Data Preprocessing

In [ ]:
df_clean = df.copy()

# Treat outliers using IQR capping
outlier_cols = ['AnnualIncome', 'TotalSpending', 'PurchaseFrequency',
                'AverageOrderValue', 'DaysSinceLastPurchase', 'WebsiteVisits']
for col in outlier_cols:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    df_clean[col] = df_clean[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# Encode Gender
le = LabelEncoder()
df_clean['Gender_Encoded'] = le.fit_transform(df_clean['Gender'])
print(f'Gender mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# One-hot encode ProductCategory
dummies = pd.get_dummies(df_clean['ProductCategory'], prefix='Category', dtype=int)
df_clean = pd.concat([df_clean, dummies], axis=1)

# Save cleaned data
df_clean.to_csv(os.path.join(BASE_DIR, 'data', 'cleaned_customer_data.csv'), index=False)
print(f'Cleaned dataset shape: {df_clean.shape}')
df_clean.head()

## Step 5: K-Means Clustering

In [ ]:
# Select RFM-based clustering features
cluster_features = ['DaysSinceLastPurchase', 'PurchaseFrequency', 'TotalSpending',
                    'AverageOrderValue', 'WebsiteVisits', 'DiscountUsage', 'CustomerRating']

X_cluster = df_clean[cluster_features]

# Scale features
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X_cluster)

print(f'Features selected for clustering: {cluster_features}')
print(f'Scaled feature matrix shape: {X_scaled.shape}')

In [ ]:
# Elbow Method
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

# Silhouette plot
optimal_k = list(k_range)[np.argmax(sil_scores)]
axes[1].plot(list(k_range), sil_scores, 'rs-', linewidth=2, markersize=8)
axes[1].axvline(x=optimal_k, color='green', linestyle='--', label=f'Optimal K={optimal_k}')
axes[1].set_title('Silhouette Score Analysis', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cluster Selection Methods', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'elbow_method.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(IMAGES_DIR, 'silhouette_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal K (by silhouette): {optimal_k} | Score: {max(sil_scores):.4f}')

In [ ]:
# Fit final K-Means model
n_clusters = max(optimal_k, 4)  # Ensure at least 4 for business relevance
final_km = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42)
df_clean['Cluster'] = final_km.fit_predict(X_scaled)

final_sil = silhouette_score(X_scaled, df_clean['Cluster'])
print(f'Final K-Means: K={n_clusters}, Silhouette Score={final_sil:.4f}')
print(f'Cluster distribution:\n{df_clean["Cluster"].value_counts().sort_index()}')

In [ ]:
# PCA 2D Cluster Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
colors_list = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA scatter
for i in range(n_clusters):
    mask = df_clean['Cluster'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_list[i % len(colors_list)], label=f'Cluster {i}',
                    alpha=0.6, s=50, edgecolors='w', linewidth=0.5)

axes[0].set_title('Customer Clusters — PCA Visualization', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend(title='Cluster')
axes[0].grid(True, alpha=0.3)

# Cluster counts
cluster_counts = df_clean['Cluster'].value_counts().sort_index()
axes[1].bar([f'Cluster {i}' for i in cluster_counts.index],
            cluster_counts.values,
            color=[colors_list[i % len(colors_list)] for i in cluster_counts.index])
for rect, count in zip(axes[1].patches, cluster_counts.values):
    axes[1].text(rect.get_x() + rect.get_width()/2, rect.get_height() + 3,
                 str(count), ha='center', fontweight='bold')
axes[1].set_title('Customers per Cluster', fontweight='bold')
axes[1].set_ylabel('Number of Customers')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'customer_clusters.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(IMAGES_DIR, 'cluster_counts.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Segment Profiling
profile_cols = ['Age', 'AnnualIncome', 'TotalSpending', 'PurchaseFrequency',
                'AverageOrderValue', 'DaysSinceLastPurchase', 'WebsiteVisits',
                'DiscountUsage', 'CustomerRating']

segment_profile = df_clean.groupby('Cluster')[profile_cols].mean().round(2)
segment_profile['CustomerCount'] = df_clean.groupby('Cluster')['Cluster'].count()
segment_profile['TotalRevenue']  = df_clean.groupby('Cluster')['TotalSpending'].sum()
segment_profile['Revenue%'] = (
    segment_profile['TotalRevenue'] / segment_profile['TotalRevenue'].sum() * 100
).round(2)

print('Segment Profiles:')
segment_profile

In [ ]:
# Save clustered data and segment profiles
df_clean.to_csv(os.path.join(OUTPUTS_DIR, 'clustered_customers.csv'), index=False)
segment_profile.to_csv(os.path.join(OUTPUTS_DIR, 'customer_segments.csv'))
print('Saved clustered_customers.csv and customer_segments.csv')

## Step 6: Classification — Purchase Likelihood (Logistic Regression)

In [ ]:
# Prepare classification data
cat_dummy_cols = [c for c in df_clean.columns if c.startswith('Category_')]
cls_features = ['Age', 'AnnualIncome', 'PurchaseFrequency', 'AverageOrderValue',
                'DaysSinceLastPurchase', 'WebsiteVisits', 'DiscountUsage',
                'CustomerRating', 'Gender_Encoded', 'Cluster'] + cat_dummy_cols
cls_features = [c for c in cls_features if c in df_clean.columns]

X_cls = df_clean[cls_features]
y_cls = df_clean['PurchaseLikelihood']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler_cls = StandardScaler()
X_train_cs = scaler_cls.fit_transform(X_train_c)
X_test_cs  = scaler_cls.transform(X_test_c)

# Train Logistic Regression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_cs, y_train_c)

y_pred_c  = clf.predict(X_test_cs)
y_prob_c  = clf.predict_proba(X_test_cs)[:, 1]

print('=== Logistic Regression — Classification Results ===')
print(f'Accuracy  : {accuracy_score(y_test_c, y_pred_c):.4f}')
print(f'Precision : {precision_score(y_test_c, y_pred_c, zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_test_c, y_pred_c, zero_division=0):.4f}')
print(f'F1-Score  : {f1_score(y_test_c, y_pred_c, zero_division=0):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test_c, y_prob_c):.4f}')
print('\nClassification Report:')
print(classification_report(y_test_c, y_pred_c, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Likely (0)', 'Likely (1)'],
            yticklabels=['Not Likely (0)', 'Likely (1)'],
            linewidths=1, linecolor='white',
            annot_kws={'size': 16, 'fontweight': 'bold'})
ax.set_title('Confusion Matrix — Purchase Likelihood', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save predictions
cls_preds = pd.DataFrame({'Actual': y_test_c.values, 'Predicted': y_pred_c, 'Probability': y_prob_c.round(4)})
cls_preds.to_csv(os.path.join(OUTPUTS_DIR, 'classification_predictions.csv'), index=False)
print('Saved classification_predictions.csv')

## Step 7: Regression — Total Spending Prediction

In [ ]:
# Prepare regression data
reg_features = ['Age', 'AnnualIncome', 'PurchaseFrequency', 'AverageOrderValue',
                'DaysSinceLastPurchase', 'WebsiteVisits', 'DiscountUsage',
                'CustomerRating', 'Gender_Encoded', 'PurchaseLikelihood'] + cat_dummy_cols
reg_features = [c for c in reg_features if c in df_clean.columns]

X_reg = df_clean[reg_features]
y_reg = df_clean['TotalSpending']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_train_rs = scaler_reg.fit_transform(X_train_r)
X_test_rs  = scaler_reg.transform(X_test_r)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_rs, y_train_r)
y_pred_lr = lr.predict(X_test_rs)

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_rs, y_train_r)
y_pred_ridge = ridge.predict(X_test_rs)

def print_reg_metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name}: MAE={mae:,.2f} | RMSE={rmse:,.2f} | R²={r2:.4f}')
    return mae, rmse, r2

print('=== Regression Results ===')
mae_lr, rmse_lr, r2_lr         = print_reg_metrics('Linear Regression', y_test_r, y_pred_lr)
mae_ridge, rmse_ridge, r2_ridge = print_reg_metrics('Ridge Regression',  y_test_r, y_pred_ridge)

In [ ]:
# Actual vs Predicted plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
all_vals = np.concatenate([y_test_r, y_pred_lr, y_pred_ridge])
mn, mx = all_vals.min(), all_vals.max()

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_ridge],
    ['Linear Regression', 'Ridge Regression'],
    ['#2196F3', '#E91E63']
):
    ax.scatter(y_test_r, y_pred, alpha=0.5, color=color, s=40)
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=2, label='Perfect Fit')
    ax.set_title(f'{title}\nActual vs Predicted', fontweight='bold')
    ax.set_xlabel('Actual Total Spending')
    ax.set_ylabel('Predicted Total Spending')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Regression: Actual vs Predicted Spending', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'regression_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save predictions
reg_preds = pd.DataFrame({
    'Actual_TotalSpending': y_test_r.values,
    'LinearRegression_Predicted': y_pred_lr.round(2),
    'Ridge_Predicted': y_pred_ridge.round(2)
})
reg_preds.to_csv(os.path.join(OUTPUTS_DIR, 'regression_predictions.csv'), index=False)
print('Saved regression_predictions.csv')

## Step 8: Hyperparameter Tuning

In [ ]:
# Tune Logistic Regression
print('Tuning Logistic Regression...')
param_grid_cls = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}
grid_cls = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_cls, cv=5, scoring='f1', n_jobs=-1
)
grid_cls.fit(X_train_cs, y_train_c)
best_clf = grid_cls.best_estimator_
y_pred_tuned = best_clf.predict(X_test_cs)

print(f'Best params: {grid_cls.best_params_}')
print(f'Baseline F1: {f1_score(y_test_c, y_pred_c, zero_division=0):.4f}')
print(f'Tuned F1   : {f1_score(y_test_c, y_pred_tuned, zero_division=0):.4f}')

In [ ]:
# Tune Ridge Regression
print('Tuning Ridge Regression...')
param_grid_reg = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
grid_reg = GridSearchCV(
    Ridge(), param_grid_reg, cv=5,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid_reg.fit(X_train_rs, y_train_r)
best_ridge = grid_reg.best_estimator_
y_pred_ridge_tuned = best_ridge.predict(X_test_rs)

rmse_tuned = np.sqrt(mean_squared_error(y_test_r, y_pred_ridge_tuned))
r2_tuned   = r2_score(y_test_r, y_pred_ridge_tuned)

print(f'Best alpha: {grid_reg.best_params_["alpha"]}')
print(f'Baseline RMSE: {rmse_ridge:,.4f} | R²: {r2_ridge:.4f}')
print(f'Tuned    RMSE: {rmse_tuned:,.4f} | R²: {r2_tuned:.4f}')

In [ ]:
# Model Comparison Table
comparison_df = pd.DataFrame([
    {'Model': 'K-Means',                  'Objective': 'Segmentation',       'Metric': 'Silhouette Score', 'Baseline': f'{final_sil:.4f}',              'Tuned': 'N/A',             'Selected': 'Yes'},
    {'Model': 'Linear Regression',         'Objective': 'Spending Prediction', 'Metric': 'RMSE / R²',       'Baseline': f'{rmse_lr:,.2f} / {r2_lr:.4f}',   'Tuned': 'N/A',             'Selected': 'No'},
    {'Model': 'Ridge Regression',          'Objective': 'Spending Prediction', 'Metric': 'RMSE / R²',       'Baseline': f'{rmse_ridge:,.2f} / {r2_ridge:.4f}', 'Tuned': f'{rmse_tuned:,.2f} / {r2_tuned:.4f}', 'Selected': 'Yes'},
    {'Model': 'Logistic Regression',       'Objective': 'Purchase Likelihood', 'Metric': 'F1 / ROC-AUC',    'Baseline': f'{f1_score(y_test_c, y_pred_c, zero_division=0):.4f} / {roc_auc_score(y_test_c, y_prob_c):.4f}', 'Tuned': f'{f1_score(y_test_c, y_pred_tuned, zero_division=0):.4f}', 'Selected': 'Yes'},
])
print('\n=== Model Comparison Table ===')
comparison_df

## Step 9: Business Insights & Recommendations

In [ ]:
# Generate business recommendations using the src module
sys.path.insert(0, BASE_DIR)
from src.business_insights import run_business_insights

# Build segment_names from profile (simplified mapping)
seg_names = {}
for cid in segment_profile.index:
    row = segment_profile.loc[cid]
    spending  = row['TotalSpending']
    frequency = row['PurchaseFrequency']
    recency   = row['DaysSinceLastPurchase']
    discount  = row['DiscountUsage']
    all_spending  = segment_profile['TotalSpending']
    all_freq      = segment_profile['PurchaseFrequency']
    all_recency   = segment_profile['DaysSinceLastPurchase']
    all_discount  = segment_profile['DiscountUsage']
    if spending >= all_spending.quantile(0.75) and frequency >= all_freq.quantile(0.75):
        seg_names[cid] = 'High-Value Loyal Customers'
    elif recency <= all_recency.quantile(0.25) and frequency <= all_freq.quantile(0.50):
        seg_names[cid] = 'New and Promising Customers'
    elif discount >= all_discount.quantile(0.75):
        seg_names[cid] = 'Discount-Driven Customers'
    elif recency >= all_recency.quantile(0.75):
        seg_names[cid] = 'At-Risk Customers'
    else:
        seg_names[cid] = 'Low-Engagement Customers'

segment_profile['SegmentName'] = segment_profile.index.map(seg_names)
segment_profile.to_csv(os.path.join(OUTPUTS_DIR, 'customer_segments.csv'))

recommendations = run_business_insights(segment_profile, seg_names, OUTPUTS_DIR)
print('Business recommendations generated!')

In [ ]:
# Display recommendations summary
rec_summary = pd.DataFrame([{
    'Cluster': cid,
    'Segment': v['icon'] + ' ' + v['segment_name'],
    'Customers': v['customer_count'],
    'Avg Spending': f"${v['avg_spending']:,.0f}",
    'Revenue %': f"{v['revenue_pct']:.1f}%",
    'KPI': v['kpi']
} for cid, v in recommendations.items()])
print('\n=== Customer Segment Recommendations ===')
rec_summary

## Step 10: Summary

### What we accomplished:

| Step | Task | Status |
|------|------|--------|
| 1 | Data Loading & Inspection | ✅ |
| 2 | Exploratory Data Analysis | ✅ |
| 3 | Data Preprocessing (encoding, scaling, outliers) | ✅ |
| 4 | K-Means Clustering (Elbow + Silhouette) | ✅ |
| 5 | Cluster Profiling & Naming | ✅ |
| 6 | Classification — Logistic Regression | ✅ |
| 7 | Regression — Linear & Ridge | ✅ |
| 8 | Hyperparameter Tuning (GridSearchCV) | ✅ |
| 9 | Business Insights & Recommendations | ✅ |
| 10 | All Visualisations & Outputs Saved | ✅ |

### Key Findings:
- **Optimal K** for clustering identified via Silhouette Score analysis
- **Purchase Likelihood** prediction achieved with Logistic Regression (tuned via GridSearchCV)
- **Total Spending** predicted with Ridge Regression (tuned alpha for best RMSE/R²)
- **Business Recommendations** generated for each customer segment in `outputs/business_recommendations.md`